# Module 3: RAG Pipeline Walkthrough

This notebook walks through the complete RAG pipeline step by step,
letting you see the intermediate results at each stage.

**Prerequisites:** Run `pip install -r requirements.txt` and create a `.env` file with your `ANTHROPIC_API_KEY`.

In [ ]:
import os
import sys
sys.path.insert(0, '..')  # Add repo root to path

from dotenv import load_dotenv
from anthropic import Anthropic
import chromadb
from chromadb.utils import embedding_functions

load_dotenv()
client = Anthropic()
print('✓ Setup complete')

## Step 1: Define Documents

In a real system these would come from PDF files, a database, or a wiki.
Here we define them directly so you can see the exact content.

In [ ]:
DOCUMENTS = [
    {
        'id':      'hr_pto_001',
        'source':  'employee_handbook.pdf',
        'page':    12,
        'content': (
            'Full-time employees receive 20 days of Paid Time Off (PTO) per year. '
            'PTO accrues at 1.67 days per month. Up to 10 unused days roll over annually.'
        ),
    },
    {
        'id':      'hr_remote_001',
        'source':  'employee_handbook.pdf',
        'page':    24,
        'content': (
            'Employees may work remotely up to 3 days per week with manager approval. '
            'Core hours are 10 AM to 3 PM local time.'
        ),
    },
    {
        'id':      'fin_expense_001',
        'source':  'finance_policy.pdf',
        'page':    5,
        'content': (
            'Expenses over $500 require Finance sign-off. '
            'Submit all expenses within 30 days via Expensify.'
        ),
    },
]

print(f'Defined {len(DOCUMENTS)} documents')
for doc in DOCUMENTS:
    print(f"  [{doc['source']}, p.{doc['page']}] {doc['content'][:60]}...")

## Step 2: Embed and Store in ChromaDB

The embedding model converts each document's text into a vector (list of numbers).
Similar texts produce similar vectors — this is what enables semantic search.

In [ ]:
# Use an in-memory ChromaDB for the notebook (no files written)
chroma = chromadb.EphemeralClient()
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='all-MiniLM-L6-v2'  # Downloads ~90MB on first run
)

collection = chroma.create_collection(
    name='notebook_demo',
    embedding_function=embed_fn
)

collection.add(
    ids       = [d['id']      for d in DOCUMENTS],
    documents = [d['content'] for d in DOCUMENTS],
    metadatas = [{'source': d['source'], 'page': d['page']} for d in DOCUMENTS],
)

print(f'✓ Embedded and stored {collection.count()} documents in ChromaDB')

## Step 3: Retrieve — Semantic Search

ChromaDB embeds the query using the same model, then finds the documents
whose vectors are closest to the query vector. Lower distance = more similar.

In [ ]:
query = 'How many vacation days do I get and can I save them for next year?'

results = collection.query(
    query_texts=[query],
    n_results=2,
    include=['documents', 'metadatas', 'distances']
)

print(f'Query: "{query}"\n')
print('Retrieved chunks (sorted by similarity):')
for doc, meta, dist in zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0]
):
    print(f'  Distance: {dist:.4f}  [{meta["source"]}, p.{meta["page"]}]')
    print(f'  Content:  {doc[:100]}...')
    print()

## Step 4: Generate — Grounded Answer with Citations

We pass the retrieved chunks to Claude as context and ask it to answer
using only that context, with citations.

In [ ]:
# Build the context block from retrieved chunks
context_block = ''
for i, (doc, meta) in enumerate(
    zip(results['documents'][0], results['metadatas'][0]), start=1
):
    context_block += f'[Source {i}: {meta["source"]}, page {meta["page"]}]\n{doc}\n\n'

system_prompt = (
    'You are a helpful HR assistant. '
    'Answer ONLY using the information in the <context> tags. '
    'Always cite your source. '
    'If the answer is not in the context, say: '
    '"I don\'t have that information in the provided documents."'
)

user_message = f'<context>\n{context_block}</context>\n\nEmployee question: {query}'

response = client.messages.create(
    model='claude-haiku-4-5-20251001',
    max_tokens=256,
    system=system_prompt,
    messages=[{'role': 'user', 'content': user_message}]
)

print('Answer:')
print(response.content[0].text)

## Step 5: Test the Fallback

Ask a question that isn't in the knowledge base.
The model should say it doesn't know — not hallucinate an answer.

In [ ]:
out_of_scope = 'What is the parental leave policy?'

results2 = collection.query(
    query_texts=[out_of_scope],
    n_results=1,
    include=['documents', 'metadatas', 'distances']
)

context2 = f'[Source 1: {results2["metadatas"][0][0]["source"]}]\n{results2["documents"][0][0]}'

response2 = client.messages.create(
    model='claude-haiku-4-5-20251001',
    max_tokens=128,
    system=system_prompt,
    messages=[{'role': 'user', 'content': f'<context>\n{context2}\n</context>\n\nEmployee question: {out_of_scope}'}]
)

print(f'Query: "{out_of_scope}"')
print(f'Best chunk distance: {results2["distances"][0][0]:.4f} (high = not relevant)')
print(f'Answer: {response2.content[0].text}')

## Exercise

1. Add 2 more documents to `DOCUMENTS` on a topic of your choice
2. Run a query that should match one of your new documents
3. Try a query that spans two documents — does the model synthesise both sources correctly?

**Next:** Run `python code/module3/lesson2_advanced_rag.py` for metadata filtering and hybrid search.